<a href="https://colab.research.google.com/github/tarlie18/Agentic_AI_101/blob/main/agentic_ai_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-google-genai ddgs

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from getpass import getpass

google_api_key = getpass("Enter your Gemini API key: ")
if not google_api_key:
    raise ValueError("No API key entered — rerun this cell and paste your key.")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    max_retries=0
)

In [ ]:
from langchain_core.tools import tool
from ddgs import DDGS
import requests
import xml.etree.ElementTree as ET

@tool
def web_search(query: str) -> str:
    """Useful for searching current events and verifying health-related rumors."""
    try:
        with DDGS(timeout=10) as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No search results found."
        return "\n".join([r["body"] for r in results])
    except Exception as e:
        return f"Search failed: {e}"

@tool
def calculator(expression: str) -> str:
    """Evaluates a basic math expression, e.g. '(120/4000)*100' for percentage calculations."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

@tool
def cdc_data(dataset_id: str, query_params: str = "") -> str:
    """Fetch data from a CDC open dataset. dataset_id is the Socrata dataset ID from data.cdc.gov."""
    url = f"https://data.cdc.gov/resource/{dataset_id}.json?{query_params}"
    resp = requests.get(url)
    return resp.text[:2000]

@tool
def news_search(query: str) -> str:
    """Search recent news articles specifically, useful for verifying if a health rumor has been reported."""
    try:
        with DDGS(timeout=10) as ddgs:
            results = list(ddgs.news(query, max_results=5))
        if not results:
            return "No news results found."
        return "\n".join([f"{r['title']}: {r['body']}" for r in results])
    except Exception as e:
        return f"News search failed: {e}"

@tool
def medlineplus_lookup(topic: str) -> str:
    """Look up authoritative health information on a disease or condition from MedlinePlus (NIH/NLM)."""
    try:
        url = "https://wsearch.nlm.nih.gov/ws/query"
        params = {"db": "healthTopics", "term": topic, "retmax": 1}
        resp = requests.get(url, params=params, timeout=10)
        root = ET.fromstring(resp.content)
        snippet = root.find(".//content[@name='snippet']")
        if snippet is not None and snippet.text:
            # Strip HTML tags MedlinePlus includes in the snippet
            import re
            clean_text = re.sub('<[^<]+?>', '', snippet.text)
            return clean_text
        return "No MedlinePlus entry found for this topic."
    except Exception as e:
        return f"MedlinePlus lookup failed: {e}"

tools = [web_search, calculator, cdc_data, news_search, medlineplus_lookup]

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are an HHS data analyst. Only output factual health data. Do not provide medical advice.
Today's date is July 2026. Trust this date as ground truth — do not question or re-verify it.

After investigating, respond ONLY with valid JSON in this exact format:
{"rumor": "...", "verified": true/false, "threat_level": "Low/Medium/High", "summary": "..."}"""
)

In [ ]:
query = "There are reports of an unusual respiratory illness in a specific zip code. Investigate and summarize."
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
)
print(result["messages"][-1].content)

In [ ]:
conversation = []

while True:
    user_query = input("You: ")
    if user_query.lower() == "quit":
        break
    conversation.append({"role": "user", "content": user_query})
    try:
        result = agent.invoke(
            {"messages": conversation},
        )
        reply = result["messages"][-1].content
        print("Agent:", reply)
        conversation.append({"role": "assistant", "content": reply})
    except Exception as e:
        print("ERROR:", e)